In [2]:
import numpy as np
import tensorflow as tf
import wave
import math
import struct

from keras.models import Sequential
from keras.layers import Dense, LSTM
from keras.utils import to_categorical


# %% [markdown]
# ## Prepare a dummy data for simulating notes


# %%
# Notes with frequencies
notes_freq = {
    # Octave 3
    'C3': 130.81,
    'C#3': 138.59,
    'D3': 146.83,
    'D#3': 155.56,
    'E3': 164.81,
    'F3': 174.61,
    'F#3': 185.00,
    'G3': 196.00,
    'G#3': 207.65,
    'A3': 220.00,
    'A#3': 233.08,
    'B3': 246.94,

    # Octave 4
    'C4': 261.63,
    'C#4': 277.18,
    'D4': 293.66,
    'D#4': 311.13,
    'E4': 329.63,
    'F4': 349.23,
    'F#4': 369.99,
    'G4': 392.00,
    'G#4': 415.30,
    'A4': 440.00,
    'A#4': 466.16,
    'B4': 493.88,

    # Octave 5
    'C5': 523.25,
    'C#5': 554.37,
    'D5': 587.33,
    'D#5': 622.25,
    'E5': 659.25,
    'F5': 698.46,
    'F#5': 739.99,
    'G5': 783.99,
    'G#5': 830.61,
    'A5': 880.00,
    'A#5': 932.33,
    'B5': 987.77
}

notes_freq


# %%
notes = list(notes_freq.keys())
notes


# %%
note_to_int = {
    note: i for i, note in enumerate(notes)
}

note_to_int


# %%
int_to_note = {
    i: note for i, note in enumerate(notes)
}

int_to_note


# %% [markdown]
# ## Generate dummy music data
#
# We will use an A minor scale:
#
# A, B, C, D, E, F, G
#
# with different octaves.


# %%
minor_notes = [
    'A3',
    'B3',
    'C4',
    'D4',
    'E4',
    'F4',
    'G4',
    'A4',
    'B4',
    'C5',
    'D5',
    'E5',
    'F5',
    'G5',
    'A5'
]

raw_music_data = [
    minor_notes[np.random.randint(0, len(minor_notes))]
    for i in range(1000)
]

raw_music_data[:50]


# %% [markdown]
# ## Data Preparation


# %%
sequence_length = 3

network_input = []
network_output = []

for i in range(len(raw_music_data) - sequence_length):

    sequence_in = raw_music_data[i:i + sequence_length]

    sequence_out = raw_music_data[i + sequence_length]

    network_input.append(
        [note_to_int[char] for char in sequence_in]
    )

    network_output.append(
        note_to_int[sequence_out]
    )

    print(sequence_in, '--->', sequence_out)


# %%
network_input


# %%
n_patterns = len(network_input)

n_patterns


# %%
x = np.reshape(
    network_input,
    (n_patterns, sequence_length, 1)
)

x.shape


# %% [markdown]
# ## Normalize input
#
# The note numbers are converted to values between 0 and 1.


# %%
x = x / float(len(notes) - 1)

x.shape


# %%
y = to_categorical(
    network_output,
    num_classes=len(notes)
)

y.shape


# %%
y


# %% [markdown]
# ## Build the LSTM model


# %%
model = Sequential()

model.add(
    LSTM(
        256,
        input_shape=(sequence_length, 1)
    )
)

model.add(
    Dense(
        1000,
        activation='relu'
    )
)

model.add(
    Dense(
        len(notes),
        activation='softmax'
    )
)

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam'
)

model.summary()


# %% [markdown]
# ## Train the model


# %%
model.fit(
    x,
    y,
    epochs=100,
    batch_size=32
)


# %% [markdown]
# ## Generate a New Melody Sequence


# %%
start_index = np.random.randint(
    0,
    len(network_input)
)

pattern = network_input[start_index].copy()

pattern


# %% [markdown]
# ## Predict


# %%
generated_melody = []

for i in range(32):

    # Normalize pattern before prediction
    x_input = np.reshape(
        pattern,
        (1, len(pattern), 1)
    )

    x_input = x_input / float(len(notes) - 1)

    # Predict next note
    prediction = model.predict(
        x_input,
        verbose=0
    )

    # Get the most likely note
    index = np.argmax(prediction)

    result = int_to_note[index]

    generated_melody.append(result)

    # Add predicted note to pattern
    pattern.append(index)

    # Keep only last 3 notes
    pattern = pattern[1:len(pattern)]


# %%
generated_melody


# %% [markdown]
# ## Save Generated Melody as WAV Audio


# %%
sample_rate = 44100

note_duration = 0.5

volume = 0.5

with wave.open('my_music.wav', 'w') as wave_file:

    # Mono audio
    # 2 bytes per sample
    # 44100 Hz
    wave_file.setparams(
        (
            1,
            2,
            sample_rate,
            0,
            'NONE',
            'not compressed'
        )
    )

    for note in generated_melody:

        freq = float(notes_freq[note])

        num_samples = int(
            note_duration * sample_rate
        )

        for i in range(num_samples):

            t = float(i) / sample_rate

            value = int(
                32767
                * volume
                * math.sin(
                    2 * math.pi * freq * t
                )
            )

            data = struct.pack(
                '<h',
                value
            )

            wave_file.writeframes(data)


print("Music saved as my_music.wav")

['A3', 'B3', 'E4'] ---> C5
['B3', 'E4', 'C5'] ---> E4
['E4', 'C5', 'E4'] ---> D4
['C5', 'E4', 'D4'] ---> D5
['E4', 'D4', 'D5'] ---> F5
['D4', 'D5', 'F5'] ---> D5
['D5', 'F5', 'D5'] ---> A4
['F5', 'D5', 'A4'] ---> B4
['D5', 'A4', 'B4'] ---> F4
['A4', 'B4', 'F4'] ---> A4
['B4', 'F4', 'A4'] ---> G4
['F4', 'A4', 'G4'] ---> F4
['A4', 'G4', 'F4'] ---> B3
['G4', 'F4', 'B3'] ---> G5
['F4', 'B3', 'G5'] ---> F5
['B3', 'G5', 'F5'] ---> C4
['G5', 'F5', 'C4'] ---> A5
['F5', 'C4', 'A5'] ---> E5
['C4', 'A5', 'E5'] ---> B4
['A5', 'E5', 'B4'] ---> C4
['E5', 'B4', 'C4'] ---> D4
['B4', 'C4', 'D4'] ---> E5
['C4', 'D4', 'E5'] ---> A5
['D4', 'E5', 'A5'] ---> A3
['E5', 'A5', 'A3'] ---> A5
['A5', 'A3', 'A5'] ---> A3
['A3', 'A5', 'A3'] ---> G4
['A5', 'A3', 'G4'] ---> A4
['A3', 'G4', 'A4'] ---> F5
['G4', 'A4', 'F5'] ---> A3
['A4', 'F5', 'A3'] ---> G5
['F5', 'A3', 'G5'] ---> B3
['A3', 'G5', 'B3'] ---> F4
['G5', 'B3', 'F4'] ---> A3
['B3', 'F4', 'A3'] ---> A3
['F4', 'A3', 'A3'] ---> A5
['A3', 'A3', 'A5'] ---> D5
[

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 256)            │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1000)           │       257,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 36)             │        36,036 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 557,228 (2.13 MB)

 Trainable params: 557,228 (2.13 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 3.0817
Epoch 2/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7665 
Epoch 3/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.7293 
Epoch 4/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.7270 
Epoch 5/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.7209 
Epoch 6/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.7227 
Epoch 7/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.7177 
Epoch 8/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7237 
Epoch 9/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7216 
Epoch 10/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7196 
Epoch 11/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7156 
Epoch 12/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7142 
Epoch 13/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.7146 
Epoch 14/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.7121 
Epoch 15/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss